# 第二十五课:SFT——教模型听指令

讲义:[25-sft.md](25-sft.md)。底座只会续写;本课用 chat 模板 + loss mask 教它「回答」,并实测灾难性遗忘与回放疗法。基座 = 第 24 课 2000 步确定性复刻。

In [ ]:
import math
import time
import numpy as np

# ===== 第 20 课 MiniGPT 原码(含 generate) =====
def softmax(x, axis=-1):
    m = np.max(x, axis=axis, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(axis=axis, keepdims=True)

class MiniGPT:
    def __init__(self, vocab=16, d=32, n_heads=4, n_layers=2, seed=0):
        rng = np.random.default_rng(seed)
        s = 0.1
        self.E = rng.normal(size=(vocab, d)) * 0.5
        self.dh = d // n_heads
        self.H, self.D, self.NL = n_heads, d, n_layers
        self.layers = [{
            'Wq': rng.normal(size=(d, d))*s, 'Wk': rng.normal(size=(d, d))*s,
            'Wv': rng.normal(size=(d, d))*s, 'Wo': rng.normal(size=(d, d))*s,
            'W1': rng.normal(size=(d, 4*d))*s*0.8, 'W2': rng.normal(size=(4*d, d))*s*0.8,
            'W3': rng.normal(size=(d, 4*d))*s*0.8,
        } for _ in range(n_layers)]
        self.head = rng.normal(size=(d, vocab)) * s

    def rms(self, x, eps=1e-5):
        return x / np.sqrt((x**2).mean(-1, keepdims=True) + eps)

    def rope(self, x, pos):
        half = x.shape[-1] // 2
        ang = pos[:, None] / (10000 ** (np.arange(half)*2/x.shape[-1]))
        c, s = np.cos(ang), np.sin(ang)
        x1, x2 = x[..., :half], x[..., half:]
        return np.concatenate([x1*c - x2*s, x1*s + x2*c], -1)

    def _attn(self, x, pos, W, cache):
        n = x.shape[0]
        q, k, v = self.rope(x@W['Wq'], pos), self.rope(x@W['Wk'], pos), x@W['Wv']
        if cache is not None:
            k, v = np.concatenate([cache[0], k]), np.concatenate([cache[1], v])
        T = k.shape[0]
        mask = np.triu(np.ones((n, T), bool), T - n + 1)
        outs = []
        for h in range(self.H):
            sl = slice(h*self.dh, (h+1)*self.dh)
            s = q[:, sl] @ k[:, sl].T / math.sqrt(self.dh)
            s = np.where(mask, -np.inf, s)
            outs.append(softmax(s) @ v[:, sl])
        return np.concatenate(outs, -1) @ W['Wo'], (k, v)

    def _ffn(self, x, W):
        h = x @ W['W1']
        return (h / (1 + np.exp(-h)) * (x @ W['W3'])) @ W['W2']

    def forward(self, ids, cache=None):
        pos0 = cache[0][0].shape[0] if cache else 0
        x = self.E[list(ids)]
        new_cache = []
        for i, W in enumerate(self.layers):
            a, kv = self._attn(x, np.arange(pos0, pos0+len(ids)), W,
                               cache[i] if cache else None)
            x = x + a
            x = x + self._ffn(self.rms(x), W)
            new_cache.append(kv)
        logits = self.rms(x) @ self.head
        assert np.isfinite(logits).all(), 'NaN! 查掩码/精度/ε'
        return logits, new_cache

    def generate(self, ids, n, T=1.0, top_p=1.0, rng=None):
        rng = rng or np.random.default_rng()
        logits, cache = self.forward(ids)
        out = []
        for _ in range(n):
            p = softmax(logits[-1] / T)
            if top_p < 1.0:
                order = np.argsort(-p); cum = np.cumsum(p[order])
                keep = order[:np.searchsorted(cum, top_p)+1]
                m = np.zeros_like(p); m[keep] = p[keep]; p = m/m.sum()
            nxt = int(rng.choice(len(p), p=p/p.sum()))
            out.append(nxt)
            logits, cache = self.forward([nxt], cache)
        return list(ids) + out

In [ ]:
# ===== 第 23 课反向 + 本课 mask 扩展(合并为一个函数;mask=None 即 23 课原版) =====
def rms_backward(dy, x, eps=1e-5):
    r = np.sqrt((x**2).mean(-1, keepdims=True) + eps)
    y = x / r
    return (dy - y * (dy * y).mean(-1, keepdims=True)) / r

def rope_backward(dy, pos, dim):
    half = dim // 2
    ang = pos[:, None] / (10000 ** (np.arange(half)*2/dim))
    c, s = np.cos(ang), np.sin(ang)
    dy1, dy2 = dy[..., :half], dy[..., half:]
    return np.concatenate([dy1*c + dy2*s, -dy1*s + dy2*c], -1)

def silu_grad(u):
    sig = 1 / (1 + np.exp(-u))
    return sig * (1 + u * (1 - sig))

def loss_and_grads(gpt, ids, targets, mask=None):
    """23 课整机反向 + 25 课可选 mask(监督位加权)。mask=None 等价 23 课原版。"""
    T = len(ids)
    if mask is None:
        mask = [1.0] * T
    marr = np.array(mask, dtype=float)
    Ns = marr.sum()
    pos = np.arange(T)
    tape = []
    x = gpt.E[list(ids)]
    for W in gpt.layers:
        q, k, v = gpt.rope(x@W['Wq'], pos), gpt.rope(x@W['Wk'], pos), x@W['Wv']
        m = np.triu(np.ones((T, T), bool), 1)
        ps, outs = [], []
        for h in range(gpt.H):
            sl = slice(h*gpt.dh, (h+1)*gpt.dh)
            s = q[:, sl] @ k[:, sl].T / math.sqrt(gpt.dh)
            s = np.where(m, -np.inf, s)
            p = softmax(s)
            ps.append(p); outs.append(p @ v[:, sl])
        co = np.concatenate(outs, -1)
        xm = x + co @ W['Wo']
        xr = gpt.rms(xm)
        u, zv = xr @ W['W1'], xr @ W['W3']
        gate = u/(1+np.exp(-u))*zv
        tape.append(dict(x_in=x, xm=xm, xr=xr, u=u, zv=zv, gate=gate, co=co, q=q, k=k, v=v, ps=ps))
        x = xm + gate @ W['W2']                 # 注意:x 入 tape 在覆盖之前(25 课战争故事②)

    xf = x
    logits = gpt.rms(xf) @ gpt.head
    z = logits - logits.max(-1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(-1, keepdims=True))
    loss = -(logp[np.arange(T), np.array(targets)] * marr).sum() / Ns

    grads = {'E': np.zeros_like(gpt.E), 'head': np.zeros_like(gpt.head),
             'layers': [dict.fromkeys(W) for W in gpt.layers]}
    p = softmax(logits)
    onehot = np.zeros_like(p)
    onehot[np.arange(T), np.array(targets)] = 1.0
    dlogits = (p - onehot) * (marr / Ns)[:, None]      # ← mask 版与 23 课的核心差别
    grads['head'] = gpt.rms(xf).T @ dlogits
    dx = rms_backward(dlogits @ gpt.head.T, xf)
    for i in reversed(range(gpt.NL)):
        W, t = gpt.layers[i], tape[i]
        dg = dx @ W['W2'].T
        du = dg * silu_grad(t['u']) * t['zv']
        dzv = dg * (t['u']/(1+np.exp(-t['u'])))
        grads['layers'][i]['W2'] = t['gate'].T @ dx
        grads['layers'][i]['W1'] = t['xr'].T @ du
        grads['layers'][i]['W3'] = t['xr'].T @ dzv
        dxm = dx + rms_backward(du @ W['W1'].T + dzv @ W['W3'].T, t['xm'])
        dc = dxm @ W['Wo'].T
        grads['layers'][i]['Wo'] = t['co'].T @ dxm
        dvs, dqs, dks = [], [], []
        for h in range(gpt.H):
            sl = slice(h*gpt.dh, (h+1)*gpt.dh)
            p_h, v_h, q_h, k_h = t['ps'][h], t['v'][:, sl], t['q'][:, sl], t['k'][:, sl]
            dO = dc[:, sl]
            dvs.append(p_h.T @ dO)
            dp = dO @ v_h.T
            ds = p_h * (dp - (dp*p_h).sum(-1, keepdims=True))
            dqs.append(ds @ k_h / math.sqrt(gpt.dh))
            dks.append(ds.T @ q_h / math.sqrt(gpt.dh))
        dv = np.concatenate(dvs, -1)
        dq = rope_backward(np.concatenate(dqs, -1), pos, gpt.D)
        dk = rope_backward(np.concatenate(dks, -1), pos, gpt.D)
        grads['layers'][i]['Wv'] = t['x_in'].T @ dv
        grads['layers'][i]['Wq'] = t['x_in'].T @ dq
        grads['layers'][i]['Wk'] = t['x_in'].T @ dk
        dx = dxm + dq @ W['Wq'].T + dk @ W['Wk'].T + dv @ W['Wv'].T
    np.add.at(grads['E'], np.array(ids), dx)
    return loss, grads

def flat_params(gpt):
    ps = [gpt.E, gpt.head]
    for W in gpt.layers:
        ps += [W[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]
    return ps

def flat_grads(grads):
    gs = [grads['E'], grads['head']]
    for L in grads['layers']:
        gs += [L[k] for k in ('Wq', 'Wk', 'Wv', 'Wo', 'W1', 'W2', 'W3')]
    return gs

def clip_norms(gs, max_norm=1.0):
    total = math.sqrt(sum((g ** 2).sum() for g in gs))
    if total > max_norm:
        s = max_norm / total
        for g in gs:
            g *= s
    return total

class MultiAdamW:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, wd=0.01):
        self.ps, self.lr, self.b1, self.b2, self.eps, self.wd = params, lr, *betas, eps, wd
        self.m = [np.zeros_like(p) for p in params]
        self.v = [np.zeros_like(p) for p in params]
        self.t = 0
    def step(self, gs):
        self.t += 1
        for p, g, m, v in zip(self.ps, gs, self.m, self.v):
            m *= self.b1; m += (1 - self.b1) * g
            v *= self.b2; v += (1 - self.b2) * g * g
            p -= self.lr * (m/(1-self.b1**self.t)/(np.sqrt(v/(1-self.b2**self.t))+self.eps) + self.wd*p)

In [ ]:
# ===== 数据 + 模板 + 任务(讲义 §3、§5) =====
text = open("data/xiyouji.txt", encoding="utf-8").read()
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
V0 = len(chars)
data = np.array([stoi[c] for c in text], dtype=np.int32)
n_train = int(len(data) * 0.99)
train_data, val_data = data[:n_train], data[n_train:]

T_USER, T_ASSI, T_END = V0, V0 + 1, V0 + 2

def enc(s):
    return [stoi[c] for c in s]

def extend_vocab(m, seed=1):
    r = np.random.default_rng(seed)
    m.E = np.vstack([m.E, r.normal(size=(3, m.D)) * 0.1])
    m.head = np.hstack([m.head, r.normal(size=(m.D, 3)) * 0.1])

ANTI = [("大","小"),("上","下"),("天","地"),("古","今"),("来","去"),("内","外"),
        ("高","低"),("长","短"),("黑","白"),("好","坏"),("真","假"),("生","死"),
        ("冷","热"),("快","慢"),("强","弱"),("轻","重"),("多","少"),("有","无"),
        ("明","暗"),("前","后"),("左","右"),("日","夜"),("开","关"),("爱","恨"),
        ("笑","哭"),("进","退"),("买","卖"),("胜","败"),("深","浅"),("忙","闲")]

COPY_STRINGS = ["金箍", "行者", "云端", "宝殿", "妖魔", "蒲团", "袈裟", "铁棒",
                "猴王", "灵山", "仙丹", "蟠桃", "佛祖", "天河", "石猴", "宝象",
                "法术", "神通", "筋斗", "火眼"]

def make_anti_ex(pair):
    q, a = pair
    ids = [T_USER] + enc(f"反义词:{q}") + [T_ASSI] + enc(a) + [T_END]
    mask = [0] * len(ids); mask[-2] = 1; mask[-1] = 1
    return ids, mask

def make_copy_ex(s):
    ids = [T_USER] + enc(f"重复:{s}") + [T_ASSI] + enc(s) + [T_END]
    mask = [0] * len(ids)
    for j in range(-len(s)-1, 0):
        mask[j] = 1
    return ids, mask

ALL_TASKS = [make_anti_ex(p) for p in ANTI] + [make_copy_ex(s) for s in COPY_STRINGS]
SPECIAL = {T_USER: '<|user|>', T_ASSI: '<|assistant|>', T_END: '<|end|>'}
def render(ids):
    return ''.join(SPECIAL[i] if i in SPECIAL else itos[i] for i in ids)
print(f"任务样本:{len(ANTI)} 反义词 + {len(COPY_STRINGS)} 复制 = {len(ALL_TASKS)}")
print("示例:", render(ALL_TASKS[0][0]), "| mask:", ALL_TASKS[0][1])

In [ ]:
# ===== masked 梯度验收(数值裁判,讲义 §4) =====
rng0 = np.random.default_rng(0)
m0 = MiniGPT(vocab=8, d=16, n_heads=2, n_layers=1, seed=3)
m0.E = np.vstack([m0.E, rng0.normal(size=(3, 16))*0.1])
m0.head = np.hstack([m0.head, rng0.normal(size=(16, 3))*0.1])
ids = [3, 4, 8, 5, 9, 10, 7]
tg, mk = ids[1:], [0, 1, 0, 0, 1, 1]
_, g0 = loss_and_grads(m0, ids[:-1], tg, mk)
gmap = {id(m0.E): g0['E'], id(m0.head): g0['head']}
for L, Lg in zip(m0.layers, g0['layers']):
    for k in L:
        gmap[id(L[k])] = Lg[k]
errs = []
rr = np.random.default_rng(1)
for arr in [m0.E, m0.head] + [W for L in m0.layers for W in L.values()]:
    for _ in range(8):
        idx = tuple(rr.integers(0, s, 1)[0] for s in arr.shape)
        old = arr[idx]
        arr[idx] = old + 1e-6; fp = loss_and_grads(m0, ids[:-1], tg, mk)[0]
        arr[idx] = old - 1e-6; fm = loss_and_grads(m0, ids[:-1], tg, mk)[0]
        arr[idx] = old
        errs.append(abs(gmap[id(arr)][idx] - (fp - fm)/2e-6))
print(f"masked 版梯度抽查 {len(errs)} 坐标 max|err| = {max(errs):.2e}")

In [ ]:
# ===== 基座:复刻 24 课 2000 步(确定性,约 2.2 分钟) =====
def lr_at(step, total, base_lr=1e-3, warmup=100, floor_ratio=0.1):
    if step < warmup:
        return base_lr * (step + 1) / warmup
    p = (step - warmup) / max(1, total - warmup)
    return base_lr * (floor_ratio + (1 - floor_ratio) * 0.5 * (1 + math.cos(math.pi * p)))

gpt = MiniGPT(vocab=V0, d=64, n_heads=4, n_layers=2, seed=0)
params = flat_params(gpt)
opt = MultiAdamW(params)
rng = np.random.default_rng(42)
T, B = 64, 16
t0 = time.time()
for step in range(2000):
    opt.lr = lr_at(step, 2000)
    off = rng.integers(0, len(train_data)-T-1, B)
    gsum, tot = None, 0.0
    for b in range(B):
        xb = train_data[off[b]:off[b]+T+1]
        loss, grads = loss_and_grads(gpt, list(xb[:-1]), list(xb[1:]))
        tot += loss
        gs = flat_grads(grads)
        if gsum is None:
            gsum = [g/B for g in gs]
        else:
            for a, g in zip(gsum, gs):
                a += g/B
    clip_norms(gsum, 1.0); opt.step(gsum)

def base_val_loss(m):
    rv = np.random.default_rng(7)
    off = rv.integers(0, len(val_data)-65, 16)
    return sum(loss_and_grads(m, list(val_data[o:o+65][:-1]),
                              list(val_data[o:o+65][1:]))[0] for o in off) / 16

bvl = base_val_loss(gpt)
print(f"基座完成({time.time()-t0:.0f}s),西游记 val loss = {bvl:.4f}")

extend_vocab(gpt)   # 扩词表:+3 特殊 token(讲义 §3)

In [ ]:
# ===== 底座不会「回答」(讲义 §2) =====
out = gpt.generate([stoi[c] for c in "一加一等于几？"], 30, T=0.8,
                   top_p=0.95, rng=np.random.default_rng(0))
print("「一加一等于几?」续写:", "".join(itos[i] for i in out))

def greedy_answer(m, prompt_ids, max_new=5):
    ids = list(prompt_ids)
    for _ in range(max_new):
        logits, _ = m.forward(ids)
        nxt = int(np.argmax(logits[-1]))
        if nxt == T_END:
            break
        ids = ids + [nxt]
    return ids

def task_acc(m):
    ok = 0; total = 0
    for q, a in [(f"反义词:{x}", y) for x, y in ANTI] + [(f"重复:{s}", s) for s in COPY_STRINGS]:
        prompt = [T_USER] + enc(q) + [T_ASSI]
        got = greedy_answer(m, prompt, len(a)+1)[len(prompt):]
        got_s = "".join(itos.get(i, "#") for i in got)
        ok += (got_s == a); total += 1
    return ok, total

print("基座任务准确率: %d/%d" % task_acc(gpt))

In [ ]:
# ===== SFT:A 纯指令 / B 半回放(讲义 §5–6) =====
import copy as _copy

def sft(m, steps, lr, replay, seed=0):
    ps = flat_params(m)
    o = MultiAdamW(ps, lr=lr)
    r = np.random.default_rng(seed)
    rp = np.random.default_rng(seed + 1)
    for step in range(steps):
        gsum, tot = None, 0.0
        n_task = B // 2 if replay else B
        batches = [(ALL_TASKS[i][0], ALL_TASKS[i][1]) for i in r.integers(0, len(ALL_TASKS), n_task)]
        if replay:
            off = rp.integers(0, len(train_data)-T-1, B - n_task)
            for o_ in off:
                xb = train_data[o_:o_+T+1]
                batches.append((list(xb[:-1]), None, list(xb[1:])))
        for item in batches:
            if len(item) == 2:      # 任务样本:mask[1:] 对齐输入位置
                ids_, mk_ = item
                loss, grads = loss_and_grads(m, ids_[:-1], ids_[1:], mk_[1:])
            else:                   # 回放样本:全监督
                ids_, _, tg_ = item
                loss, grads = loss_and_grads(m, ids_, tg_)
            tot += loss
            gs = flat_grads(grads)
            if gsum is None:
                gsum = [g/len(batches) for g in gs]
            else:
                for a, g in zip(gsum, gs):
                    a += g/len(batches)
        clip_norms(gsum, 1.0); o.step(gsum)
        if (step+1) % 250 == 0:
            print(f"    step {step+1}: loss {tot/len(batches):.4f}")

print("[SFT-A] 纯指令,lr=1e-4,1000 步")
gA = _copy.deepcopy(gpt)
sft(gA, 1000, 1e-4, replay=False)
print(f"  任务准确率 {task_acc(gA)[0]}/50 | 西游记 val {bvl:.4f} → {base_val_loss(gA):.4f}")

print("[SFT-B] 半指令 + 半西游记回放,lr=1e-4,1000 步")
gB = _copy.deepcopy(gpt)
sft(gB, 1000, 1e-4, replay=True)
print(f"  任务准确率 {task_acc(gB)[0]}/50 | 西游记 val {bvl:.4f} → {base_val_loss(gB):.4f}")

print("\n[三模型对照]「那猴王」续写(T=0.8,同种子):")
for name, m in [("基座", gpt), ("SFT-A", gA), ("SFT-B", gB)]:
    out = m.generate([stoi[c] for c in "那猴王"], 30, T=0.8,
                     top_p=0.95, rng=np.random.default_rng(1))
    print(f"  [{name}] " + "".join(itos.get(i, '#') for i in out))

print("\n[贪心回答]")
for q in ["反义词:大", "重复:筋斗"]:
    prompt = [T_USER] + enc(q) + [T_ASSI]
    for name, m in [("基座", gpt), ("SFT-A", gA)]:
        got = greedy_answer(m, prompt, 5)[len(prompt):]
        print(f"  [{name}] {q} → " + repr("".join(itos.get(i, '<|end|>') for i in got)))

In [ ]:
# ---- 作业现场 ----

# TODO(C1): 写通版 make_ex(q, a):任意问答对 → (ids, mask),mask 只监督答案与 <|end|>

# TODO(C3): 验证扩词表前后旧 id 的 logits 逐位不变
# (需要在 extend_vocab 前保存 logits,扩完后 forward 对照)

# TODO(D2): SFT 后线上永远输出空回答——按讲义 §7 的顺序写排查脚本

# TODO(E1): replay 比例 0 / 0.25 / 0.5 / 0.75 / 1 扫描,画任务准确率 vs val loss 权衡曲线

# TODO(E2/预习): 若 r=4,LoRA 挂在全部 7 个 d×d 投影上,可训练参数是多少?
#                占 713,600 的百分之几?
